# System Predykcyjny PLON Market 🛒
Poniższy notatnik przetwarza dane historyczne, wykorzystuje algorytm XGBoost do prognozy popytu i generuje listę rekomendacji dla wybranego sklepu na najbliższe 7 dni.

In [ ]:
import pandas as pd
import os
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
from itertools import product

print("⏳ Inicjalizacja systemu predykcyjnego...")

### 1. Wczytywanie i czyszczenie danych z plików CSV
Pobieramy historię sprzedaży, wydarzenia, promocje oraz cennik konkurencji.

In [ ]:
def load_and_clean(filename):
    path = os.path.join('assets', filename) if os.path.exists(os.path.join('assets', filename)) else filename
    df = pd.read_csv(path, sep=';')
    df.columns = df.columns.str.strip() 
    return df

sales = load_and_clean('Sprzedaż_dzienna.csv')
events = load_and_clean('Eventy_lokalne.csv')
promos = load_and_clean('Promocje.csv')
ceny_df = load_and_clean('Konkurencja_-_ceny.csv')

# Wyciągamy ostatnią cenę PLON z cennika
ceny_aktualne = ceny_df.sort_values('Data początku tygodnia').groupby('SKU')['Cena PLON (PLN)'].last()

sales['Data'] = pd.to_datetime(sales['Data'])
events['Data'] = pd.to_datetime(events['Data'])
promos['Data od'] = pd.to_datetime(promos['Data od'])
promos['Data do'] = pd.to_datetime(promos['Data do'])

# Naprawa identyfikatorów sklepów
sales['ID Sklepu'] = sales['ID Sklepu'].astype(str).str.replace('PLN-', '', regex=False).astype(int)

# Ograniczenie asortymentu do 'Fresh'
fresh_categories = ['Warzywa i owoce', 'Nabiał i jaja', 'Mięso', 'Pieczywo']
fresh_sales = sales[sales['Kategoria'].isin(fresh_categories)].copy()

sku_meta = fresh_sales[['ID_SKU', 'Nazwa SKU', 'Kategoria']].drop_duplicates().set_index('ID_SKU')

### 2. Inżynieria Cech (Feature Engineering)
Budujemy zestaw cech zorientowanych w czasie.

In [ ]:
promos_daily = []
for _, row in promos.iterrows():
    dates = pd.date_range(row['Data od'], row['Data do'])
    for d in dates:
        promos_daily.append({'Data': d, 'Kategoria': row['Kategoria'], 'Is_Promo': 1})
promos_df = pd.DataFrame(promos_daily).drop_duplicates()

train_df = fresh_sales.copy()
train_df = train_df.merge(events[['Data', 'Wpływ szacowany (1-5)']], on='Data', how='left')
train_df = train_df.merge(promos_df, on=['Data', 'Kategoria'], how='left')
train_df.fillna({'Is_Promo': 0, 'Wpływ szacowany (1-5)': 0}, inplace=True)

def add_features(df_feat):
    df_feat['day_of_week'] = df_feat['Data'].dt.dayofweek
    df_feat['month'] = df_feat['Data'].dt.month
    df_feat['is_weekend'] = (df_feat['day_of_week'] >= 5).astype(int)
    df_feat['ID_SKU'] = df_feat['ID_SKU'].astype('category')
    df_feat['ID Sklepu'] = df_feat['ID Sklepu'].astype('category')
    return df_feat

train_df = add_features(train_df)
features = ['day_of_week', 'month', 'is_weekend', 'ID_SKU', 'ID Sklepu', 'Is_Promo', 'Wpływ szacowany (1-5)']
target = 'Sztuki sprzedane'

### 3. Trenowanie Modelu ML (XGBoost)
Model wykorzystuje optymalizację do błędu MAE i logarytmowanie w celu obniżenia końcowego MAPE.

In [ ]:
split_date = train_df['Data'].max() - pd.Timedelta(days=7)
train_mask = train_df['Data'] <= split_date
val_mask = train_df['Data'] > split_date

X_train, y_train = train_df[train_mask][features], train_df[train_mask][target]
X_val, y_val = train_df[val_mask][features], train_df[val_mask][target]

print("🚀 Trenowanie modelu ML...")
model = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
    random_state=42, enable_categorical=True, objective='reg:absoluteerror'
)

model.fit(X_train, np.log1p(y_train))
val_preds = np.maximum(0, np.expm1(model.predict(X_val)))
mape = mean_absolute_percentage_error(y_val, val_preds)
pewnosc_str = f"Średni błąd modelu (MAPE) to {mape:.1%}"
print(pewnosc_str)

# Douczanie na całości zbioru przed generowaniem przyszłości
model.fit(train_df[features], np.log1p(train_df[target]))

### 4. Generowanie Siatki Predykcyjnej (7 Dni) i Mapowanie Cennika

In [ ]:
ostatnia_data = train_df['Data'].max()
next_7_days = pd.date_range(ostatnia_data + pd.Timedelta(days=1), periods=7)

unique_stores = fresh_sales['ID Sklepu'].unique()
unique_skus = fresh_sales['ID_SKU'].unique()

grid = list(product(next_7_days, unique_stores, unique_skus))
future_df = pd.DataFrame(grid, columns=['Data', 'ID Sklepu', 'ID_SKU'])

future_df['Kategoria'] = future_df['ID_SKU'].map(sku_meta['Kategoria'])
future_df = future_df.merge(events[['Data', 'Wpływ szacowany (1-5)']], on='Data', how='left')
future_df = future_df.merge(promos_df, on=['Data', 'Kategoria'], how='left')
future_df.fillna({'Is_Promo': 0, 'Wpływ szacowany (1-5)': 0}, inplace=True)
future_df = add_features(future_df)

future_preds_log = model.predict(future_df[features])
future_df['Przewidywana_Sprzedaz'] = np.expm1(future_preds_log)
future_df['Przewidywana_Sprzedaz'] = future_df['Przewidywana_Sprzedaz'].apply(lambda x: max(0, int(np.ceil(x))))

future_df['Nazwa produktu'] = future_df['ID_SKU'].map(sku_meta['Nazwa SKU'])

# Słownik mapujący różnice w nazwach pomiędzy plikami
name_mapping = {
    'Jabłka 1kg': 'Jabłka Ligol 1kg',
    'Masło 200g': 'Masło ekstra 200g',
    'Chleb pszenny 500g': 'Chleb wiejski 500g'
}

future_df['Nazwa do cennika'] = future_df['Nazwa produktu'].astype(str).replace(name_mapping)
future_df['Cena jednostkowa'] = future_df['Nazwa do cennika'].map(ceny_aktualne)

# Imputacja (zastępowanie) braków w cenniku średnią wyciągniętą z kategorii
srednie_kategorii = future_df.groupby('Kategoria')['Cena jednostkowa'].transform('mean')
future_df['Cena jednostkowa'] = future_df['Cena jednostkowa'].fillna(srednie_kategorii).fillna(6.0)

future_df['Łączny koszt (PLN)'] = (future_df['Przewidywana_Sprzedaz'] * future_df['Cena jednostkowa']).round(2)

### 5. Wynik Końcowy (Filtracja Wybranego Sklepu i Eksport do CSV)
Znalezienie 10 najtańszych zamówień per dzień dla zdefiniowanej placówki.

In [ ]:
def przypisz_jednostke(kat):
    if kat == 'Mięso': return 'kg'
    elif kat in ['Warzywa i owoce']: return 'kg/szt.'
    else: return 'szt.'

future_df['Jednostka'] = future_df['Kategoria'].apply(przypisz_jednostke)
future_df['Zalecana ilość do zamówienia'] = future_df['Przewidywana_Sprzedaz'].astype(str) + ' ' + future_df['Jednostka']
future_df['Pewność predykcji'] = pewnosc_str

# USTAWIENIA SKLEPU
WYBRANY_SKLEP = 20

future_df_filtered = future_df[future_df['ID Sklepu'] == WYBRANY_SKLEP].copy()

# Sortowanie rosnąco po łącznym koszcie
filtered_df = (
    future_df_filtered.sort_values(['Data', 'Łączny koszt (PLN)'], ascending=[True, True])
    .groupby('Data')
    .head(10)
)

output_columns = ['Data', 'ID Sklepu', 'Nazwa produktu', 'Kategoria', 'Zalecana ilość do zamówienia', 'Łączny koszt (PLN)', 'Pewność predykcji']
final_output = filtered_df[output_columns].copy()
final_output['Łączny koszt (PLN)'] = final_output['Łączny koszt (PLN)'].astype(str) + ' PLN'

out_path = os.path.join('assets', f'rekomendacje_zamowien_sklep_{WYBRANY_SKLEP}.csv') if os.path.exists('assets') else f'rekomendacje_zamowien_sklep_{WYBRANY_SKLEP}.csv'
final_output.to_csv(out_path, index=False, sep=';', encoding='utf-8-sig')

print(f"✅ Sukces! Plik CSV wygenerowany dla sklepu {WYBRANY_SKLEP}.")

# Podgląd końcowej tabeli prosto w notatniku
final_output.head(10)